In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import pyautogui

In [ ]:
start_time = time.time()
website = "https://efdsearch.senate.gov/"

# Configurar opciones para suprimir avisos
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-logging"])
options.add_argument("--log-level=3")

browser = webdriver.Chrome(ChromeDriverManager().install(), options=options)

browser.maximize_window()
browser.get(website)

In [3]:

# Example Selenium operations with updated syntax
browser.find_element(By.XPATH, "//*[@id='agree_statement']").click()  # Agree to terms
time.sleep(1)

# Set search criteria (e.g., searching for senate periodic transactions)
browser.find_element(By.XPATH, "//*[contains(concat(' ', @class, ' '), concat(' ', 'form-check-input', ' '))]").click()
browser.find_element(By.XPATH, "//*[@id='reportTypeLabelPtr']").click()
browser.find_element(By.XPATH, '/html/body/div[1]/main/div/div/div[5]/div/form/fieldset[4]/div/div/div/div[1]/span/input').send_keys('07/30/2021')

time.sleep(1)

# Hit the search button
browser.find_element(By.XPATH, "//*[@id='searchForm']/div/button").click()

In [4]:
data = {'Name': [], 'Transaction Date': [], 'Owner': [], 'Ticker': [],
        'Asset Name': [], 'Asset Type': [], 'Type': [], 'Amount': [], 'Comment': []}
df = pd.DataFrame(data)

In [5]:
time.sleep(1)
pagenumber = (browser.find_element(By.XPATH,'//*[contains(concat( " ", @class, " " ), concat( " ", "current", " " ))]')).text
pagenumber = int(pagenumber)
countpp = browser.find_element(By.XPATH, "/html/body/div[1]/main/div/div/div[6]/div/div/div/div[3]/div[1]/div/label/select/option[4]").click()

In [ ]:
while pagenumber <= 4:    # Will stop scraping once page 63 ends. Page 64 will need to be manually added.
    print("--- %s seconds ---" % (time.time() - start_time))
    count = 1
    
    while count <= 100:     # Should be while count <= 25 to go through entire page. 25 is the number of entries per page.
        time.sleep(2)
        caps = str(browser.find_element(By.XPATH,'//*[@id="filedReports"]/tbody/tr[' + str(count) + ']/td[1]').text)
        testcaps = (caps.isupper())
        if testcaps == False:
            button = browser.find_element(By.XPATH,'/html/body/div[1]/main/div/div/div[6]/div/div/div/table/tbody/tr[' + str(count) + ']/td[4]/a')
            browser.execute_script("arguments[0].click();", button)
            time.sleep(2)
            browser.switch_to.window(browser.window_handles[1])
            # Making a loop to sort through each individual report
            individual = 1
            while individual <= int(
                    browser.find_element(By.XPATH,'//*[@id="content"]/div/div/section/div/div/table/tbody/tr[1]/td[1]').text):
                # 36) Finds the number of transactions per report and cycles until individual hits that number.
                df.loc[-1] = {'Name': browser.find_element(By.XPATH,'//*[@id="content"]/div/div/div[2]/div[1]/h2').text,
                              'Transaction Date': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(individual) + ']/td[2]').text,
                              'Owner': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[3]').text,
                              'Ticker': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[4]').text,
                              'Asset Name': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[5]').text,
                              'Asset Type': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[6]').text,
                              'Type': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[7]').text,
                              'Amount': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[8]').text,
                              'Comment': browser.find_element(By.XPATH,
                                  '//*[@id="content"]/div/div/section/div/div/table/tbody/tr[' + str(
                                      individual) + ']/td[9]').text
                              }
                individual = individual + 1
                df.index = df.index + 1
                df = df.sort_index()
            browser.close()
            browser.switch_to.window(browser.window_handles[0])
            # Print the output.
            print(df)
            count = count + 1
        else:
            count = count + 1
    browser.switch_to.window(browser.window_handles[0])
    browser.find_element(By.XPATH,'/html/body/div[1]/main/div/div/div[6]/div/div/div/div[3]/div[2]/div/a[2]').click()
    pagenumber = pagenumber+1

--- 6.436771631240845 seconds ---
                                               Name Transaction Date Owner  \
0  The Honorable Michael F Bennet (Bennet, Michael)       10/04/2022  Self   

  Ticker                                         Asset Name        Asset Type  \
0     --  More\nCompany: More  (Ebene, Muritius (India))...  Other Securities   

       Type            Amount  \
0  Purchase  $1,001 - $15,000   

                                             Comment  
0  Underlying asset of Madeira Capital Investment...  
                                               Name Transaction Date Owner  \
0  The Honorable Michael F Bennet (Bennet, Michael)       04/26/2023  Self   
1  The Honorable Michael F Bennet (Bennet, Michael)       10/04/2022  Self   

  Ticker                                         Asset Name        Asset Type  \
0     --  More\nCompany: More  (Ebene, Muritius (India))...  Other Securities   
1     --  More\nCompany: More  (Ebene, Muritius (India))...  Other Secur

In [ ]:
df

In [ ]:
print("--- %s seconds ---" % (time.time() - start_time))
df.to_csv(r'C:\Users\student\Documents\Python\CongressInvestments.csv', index=False)